# Correcting surface currents with drifter trajectories

**Companion notebook — variational assimilation of Lagrangian drifter data into altimetric surface currents**

This notebook reproduces one assimilation experiment from start to finish. Virtual drifters are released in a *truth* field (an ocean reanalysis), and their trajectories serve as observations. A smooth correction $\delta\mathbf{u}$ to a *background* field (satellite altimetry) is then fitted by gradient descent, back-propagating through a fourth-order Runge–Kutta advection scheme, so that drifters advected in the corrected field follow the observed paths. The method follows Issa et al. (2016).

Two ways of distributing the correction in time are compared:

| | **Constant correction per sub-window** | **Sliding window** |
|---|---|---|
| Correction | one constant $\delta\mathbf{u}_w$ per sub-window $w$ | one $\delta\mathbf{u}_k$ per overlapping window $k$ (length $T_w$, shift $\sigma$) |
| Field used | $\mathbf{u}_b + \delta\mathbf{u}_w$ inside sub-window $w$ | $\mathbf{u}_b(t) + \sum_k \omega_k(t)\,\delta\mathbf{u}_k$, weights $\omega_k \propto 1/(\lvert k-k^\star(t)\rvert+1)$ (Eq. 5) |
| Code | `analysis_nDrif.multi_analysis_optm` | `sliding_window_assim.multi_analysis_sliding_window` |

For each window, the minimised cost is

$$
J(\delta\mathbf{u}) \;=\; \sum_{d=1}^{N_d} \operatorname{MSE}\!\Big(\mathbf{x}_d\big[\mathbf{u}_b + F(\delta\mathbf{u})\big],\; \mathbf{x}_d^{\mathrm{obs}}\Big)
\;+\; \alpha\,\big(\lVert\delta u\rVert + \lVert\delta v\rVert\big) \;+\; \beta\,\big\lVert \nabla\!\cdot\delta\mathbf{u} \big\rVert ,
$$

where $F$ is a diffusion filter with length scale $l_{\mathrm{fil}}$ and $\mathbf{x}_d$ is the trajectory of drifter $d$. The skill reported is the relative reduction of the velocity-field RMSE with respect to the truth.

> L. Issa, J. Brajard, M. Fakhri, D. Hayes, L. Mortier and P.-M. Poulain (2016). *Modelling surface currents in the eastern Levantine Mediterranean using surface drifters and satellite altimetry.* Ocean Modelling 104, 1–14.

**How to use.** Edit only the configuration cell (§1), then run all cells. Cells marked **expensive** run the optimisation. They are skipped automatically when results for exactly the same configuration already exist in the experiment's HDF5 file, so the figures can be restyled and re-rendered in seconds. All assimilation code is imported from `src/`. The plotting helpers live in `notebooks/assim_nb/`.

| Section | Content | Cost |
|---|---|---|
| 0 | Input data and how to run | — |
| 1 | Configuration | cheap |
| 2 | Setup and preprocessing of the raw data | cheap (preprocessing, first run only: moderate) |
| 3 | Velocity fields and release points (Fig. 1) | cheap |
| 4 | Synthetic drifter observations | cheap–moderate |
| 5 | Assimilation, both methods | **expensive** |
| 6 | Diagnostics: error in time (Fig. 2), cost decay (Fig. 3) | cheap |
| 7 | Movies of fields and trajectories | moderate |

## 0 · Input data and how to run

**Input files.** The assimilation reads two files of daily surface velocity from `data/`. §2b builds them from the raw downloads on the first run (they are git-ignored). Both hold `ugos`, `vgos` on the same 1/8° grid, with dimensions `(time, latitude, longitude)`, and cover the same dates:

| File | Role | Origin |
|---|---|---|
| `data/back_stand.nc` | background | geostrophic surface currents from a gridded (L4) satellite-altimetry product, e.g. the Copernicus Marine (CMEMS) multi-mission sea-level product at 0.125° |
| `data/truth_inter.nc` | truth | surface currents from an ocean reanalysis, e.g. the CMEMS Mediterranean physics reanalysis, interpolated onto the altimetry grid |

**Raw downloads.** The repository includes the two raw CMEMS downloads (1–31 July 2006, Mediterranean Sea), compressed in `data/raw/raw_data.zip`. **Unzip it before the first run** (see Steps). §2b builds the input files from them: it renames the velocity variables to `ugos`/`vgos`, drops the depth axis and interpolates the reanalysis onto the altimetry grid. The raw files are read from the folder in the environment variable `DRIFTERS_RAW_DIR` (default: `data/raw/` in the repository). Their file names are set by `raw_truth` and `raw_back` in §1. The configuration check in §2 stops with the name of any missing file.

**Steps.**
1. Unzip the raw data into `data/raw/`, once. From the repository root:
   ```bash
   unzip data/raw/raw_data.zip -d data/raw
   ```
   Without `unzip`, use `python -m zipfile -e data/raw/raw_data.zip data/raw`, or extract it with your file manager.
2. To use other raw downloads, put them in `data/raw/`, or point `DRIFTERS_RAW_DIR` at the folder holding them, and delete the two built files in `data/`. Set the variable in the shell that starts Jupyter, so the kernel inherits it:
   ```bash
   export DRIFTERS_RAW_DIR=/path/to/raw/downloads
   ```
3. Edit the configuration cell (§1): file names, domain, period, release points and parameters.
4. Start Jupyter from `notebooks/` and run all cells. Or run it headless from the repository root:
   ```bash
   jupyter nbconvert --to notebook --execute notebooks/assimilation.ipynb --output-dir <folder>
   ```
5. Clear the outputs before committing (see §8).


## 1 · Configuration

Every path and parameter of the experiment is set here, and nowhere else. Paths are relative to the repository, so run the notebook from `notebooks/`.

* **Domain and period.** `lon`, `lat` are the box bounds in degrees. `start_day` is the index of the first daily field in the input files. `n_days` is the assimilation length, which needs `n_days + 1` daily fields.
* **Drifters.** `lon0`, `lat0` list one release point per drifter. Any number of drifters is assimilated together.
* **Cost function and optimiser.** `lfil` is the filter length scale in metres. `alpha` and `beta` weight the correction-size and divergence penalties. `mylr` is the Adam learning rate. Iterations stop when the gradient norm changes by less than `toll` percent, or when the loss plateaus.
* **Method-specific.** `n_corr` sub-windows for the constant correction (it must divide `n_days`). Window length `tw_days` and shift `sigma_days` for the sliding window.

In [ ]:
import os
from pathlib import Path
from assim_nb.config import Config

# --- Paths -------------------------------------------------------------------
ROOT = Path.cwd().resolve().parent          # repository root (notebook runs from notebooks/)
DATA_DIR = ROOT / "data"                    # processed input NetCDF files (git-ignored)
RAW_DIR = Path(os.environ.get("DRIFTERS_RAW_DIR", ROOT / "data" / "raw"))  # raw CMEMS downloads
OUTPUT_DIR = ROOT / "output" / "notebooks"  # HDF5 results, logs, figures, movies

cfg = Config(
    root=ROOT, data_dir=DATA_DIR, output_dir=OUTPUT_DIR,
    truth_file=DATA_DIR / "truth_inter.nc",  # truth: reanalysis interpolated on the altimetry grid
    back_file=DATA_DIR / "back_stand.nc",        # background: standardised altimetry (ugos, vgos)
    raw_truth=RAW_DIR / "(new_reanalysis)cmems_mod_med_phy-cur_my_4.2km_P1D-m_1767810181825.nc",  # raw CMEMS files, standardised in §2 if the
    raw_back=RAW_DIR / "(new_aviso)cmems_obs-sl_glo_phy-ssh_my_allsat-l4-duacs-0.125deg_P1D_1767634358778.nc",     # two files above are missing
    exp_name="levantine_3days_2drifters",

    # Domain and period
    lon=[30.0, 32.0], lat=[33.0, 35.0],
    start_day=0, n_days=3,

    # Release points: one entry per drifter
    lon0=[30.3, 31.0],
    lat0=[34.3, 33.7],

    # Time stepping (s); observations are sampled at every advection step
    dt_advection=1800.0, dt_observation=1800.0,

    # Cost function and optimiser
    lfil=15000, mylr=0.01, alpha=0.001, beta=0.1, toll=1.0,

    # Constant correction per sub-window
    n_corr=3,
    # Sliding window (Issa et al. 2016)
    tw_days=1, sigma_days=1,

    # Run control
    methods=("standard", "sliding"),
    rerun=False,       # True: recompute even if matching results exist
    device="auto",     # CUDA when available, otherwise CPU
    verbose=False,     # False: per-iteration output of src/ goes to a log file

    # Figures and movies
    frame_every=3, fps=8, save_figures=True, save_movies=True, movie_format="mp4",
)

## 2 · Setup

Adds `src/` to the import path, selects the device and checks the configuration. **Cost: cheap.**

In [ ]:
%matplotlib inline
%config InlineBackend.figure_format = "retina"

import platform
import time

import h5py
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from IPython.display import display

from assim_nb.config import add_src_to_path, select_device, check_config, check_release_points, src_output
add_src_to_path(cfg.root)

import torch
from analysis_nDrif import preprocessing, velocity_lon_lat, run_onTruth_new, multi_analysis_optm, field_int_new
from sliding_window_assim import multi_analysis_sliding_window

from assim_nb import results as R, plots as P, movies as M, style as S

check_config(cfg)
device = select_device(cfg.device)
S.apply_style()
cfg.output_dir.mkdir(parents=True, exist_ok=True)

print(f"Python {platform.python_version()} · torch {torch.__version__} · matplotlib {matplotlib.__version__}"
      f" · cartopy {'yes' if S.HAS_CARTOPY else 'not installed (plain lon/lat axes)'}")
print(f"Device: {device}  ·  results: {cfg.h5_path.relative_to(cfg.root)}  ·  fingerprint {cfg.fingerprint()}")

### 2b · Preprocessing

Builds the two input files from the raw CMEMS files given in the configuration. Unzip `data/raw/raw_data.zip` first (§0). It is skipped when both input files already exist, so it only runs the first time. `preprocessing` standardises both files to `ugos`/`vgos` on a common grid and interpolates the reanalysis onto the altimetry grid. **Cost: moderate to expensive** (reads the whole raw files).

In [ ]:
cfg.data_dir.mkdir(parents=True, exist_ok=True)
if not (cfg.truth_file.exists() and cfg.back_file.exists()):
    # check_config (§2) has already stopped the run if the raw files are missing.
    preprocessing(cfg.raw_truth, cfg.raw_back, cfg.data_dir / "truth_stand.nc", cfg.truth_file, cfg.back_file)
    print("Standardised files written to", cfg.data_dir.relative_to(cfg.root))
else:
    print("Input files found.")

## 3 · Velocity fields and release points

Daily truth and background fields are cropped to the domain. Tensors are stored as `(time, lon, lat)`. Note that `velocity_lon_lat` returns the **background first**. Release points are checked to be over water and away from the boundary. **Cost: cheap.**

In [ ]:
Ub, Vb, Ut, Vt, lon, lat = velocity_lon_lat(cfg.lon, cfg.lat, cfg.start_day, cfg.end_day,
                                            cfg.truth_file, cfg.back_file)
Ut, Vt, Ub, Vb, lon, lat = (x.to(device) for x in (Ut, Vt, Ub, Vb, lon, lat))
assert Ut.shape[0] == cfg.n_days + 1, f"expected {cfg.n_days + 1} daily fields, got {Ut.shape[0]}"

maskuu = np.isnan(Ut[0].cpu().numpy())   # land mask used by the filter
maskvv = np.isnan(Vt[0].cpu().numpy())
check_release_points(cfg, R.to_numpy(lon), R.to_numpy(lat), maskuu)

lon_np, lat_np = R.to_numpy(lon), R.to_numpy(lat)
Ut_np, Vt_np, Ub_np, Vb_np = (R.to_numpy(x) for x in (Ut, Vt, Ub, Vb))
print(f"{Ut.shape[0]} daily fields on a {lon.numel()} × {lat.numel()} grid "
      f"({lon_np[1] - lon_np[0]:.3f}°), {int(maskuu.sum())} land cells, {cfg.n_drif} drifters")

In [ ]:
fig = P.plot_domain_overview(cfg, lon_np, lat_np, Ub_np, Vb_np, Ut_np, Vt_np)
S.save_figure(fig, cfg, "fig1_domain")
plt.show()

**Figure 1.** Time-mean surface currents over the assimilation period. Grey shading is speed, and arrows show the direction and magnitude of the flow. Sand-coloured cells are land. (a) Background (altimetry), which the assimilation corrects. (b) Truth (reanalysis), in which the synthetic drifters are released. Stars mark the numbered release points.

## 4 · Synthetic drifter observations

Each drifter is advected in the **truth** field at `dt_advection` resolution. The tracks are stored in the group `metadata` of the experiment's HDF5 file and serve as observations for both methods. To assimilate real drifters, write their positions into the same datasets (`lon_obs0{n}_{n_days}days`, shape `(n_days, steps_per_day)`). **Cost: cheap to moderate.**

In [ ]:
h5 = h5py.File(cfg.h5_path, "a")
obs_grp = h5.require_group("metadata")
with src_output(cfg, "observations"):
    run_onTruth_new(Ut, Vt, lon, lat, cfg.n_days, obs_grp, cfg.dt_advection,
                    cfg.lon0, cfg.lat0, cfg.n_drif, device=device)
obs = R.load_obs(h5, cfg)
print(f"{cfg.n_drif} trajectories × {obs[0].shape[1]} positions "
      f"({cfg.steps_per_day} per day, release point included)")

## 5 · Assimilation

Each method writes its corrections, cost-function history, background and analysis trajectories, and field errors into its own HDF5 group (`standard/`, `sliding/`). Both functions take all drifters at once: the cost sums the trajectory misfits of every drifter.

### 5a · Constant correction per sub-window

The assimilation period is split into `n_corr` sub-windows. Each gets its own constant correction, fitted to the trajectory segments inside it. **Cost: expensive.**

In [ ]:
if "standard" in cfg.methods:
    if R.has_results(h5, "standard", cfg) and not cfg.rerun:
        print("Reusing stored results (set rerun=True to recompute).")
    else:
        grp = h5.require_group("standard")
        t0 = time.perf_counter()
        with src_output(cfg, "standard") as log:
            rel = multi_analysis_optm(
                Ut, Vt, Ub, Vb, lon, lat, cfg.lon0, cfg.lat0, maskuu, maskvv, grp, obs_grp,
                cfg.n_days, cfg.lfil, cfg.dt_advection, cfg.dt_observation,
                cfg.mylr, cfg.alpha, cfg.beta, cfg.n_drif, cfg.n_corr, cfg.toll,
                synth_traj=True, device=device)
        R.mark_results(h5, "standard", cfg, time.perf_counter() - t0)
        h5.flush()
        print(f"Relative ΔRMSE = {rel:.2f} %  ·  {time.perf_counter() - t0:.0f} s"
              + (f"  ·  log: {log.relative_to(cfg.root)}" if log else ""))

### 5b · Sliding window

Corrections are fitted on overlapping windows of `tw_days` days, shifted by `sigma_days`. They are then blended into a time-continuous correction with weights $1/(\lvert k-k^\star\rvert+1)$ (Eq. 5 of Issa et al. 2016). The blended field $\mathbf{u}_c(t)$ is returned at every advection step and stored for the figures. **Cost: expensive.**

In [ ]:
if "sliding" in cfg.methods:
    if R.has_results(h5, "sliding", cfg) and not cfg.rerun:
        print("Reusing stored results (set rerun=True to recompute).")
    else:
        grp = h5.require_group("sliding")
        t0 = time.perf_counter()
        with src_output(cfg, "sliding") as log:
            rel, Uc, Vc = multi_analysis_sliding_window(
                Ut, Vt, Ub, Vb, lon, lat, cfg.lon0, cfg.lat0, maskuu, maskvv, grp, obs_grp,
                cfg.n_days, cfg.lfil, cfg.dt_advection, cfg.dt_observation,
                cfg.mylr, cfg.alpha, cfg.beta, cfg.n_drif, cfg.tw_days, cfg.sigma_days, cfg.toll,
                synth_traj=True, device=device)
        R.save_fields(h5, "sliding", Uc=Uc, Vc=Vc)
        R.mark_results(h5, "sliding", cfg, time.perf_counter() - t0)
        h5.flush()
        print(f"Relative ΔRMSE = {rel:.2f} %  ·  {time.perf_counter() - t0:.0f} s"
              + (f"  ·  log: {log.relative_to(cfg.root)}" if log else ""))

## 6 · Diagnostics

**Cost: cheap.** These cells only read the HDF5 file.

In [ ]:
rms = {m: R.load_rms(h5, m, cfg) for m in cfg.methods}
spans = {m: R.window_spans(cfg, m, R.n_windows(h5, m, cfg)) for m in cfg.methods}
obj = {m: R.load_objective(h5, m, cfg) for m in cfg.methods}
display(P.summary_table(cfg, h5))

**Table 1.** Skill and cost of each method. *Relative ΔRMSE* is the time mean of $(\mathrm{RMSE}_b - \mathrm{RMSE}_a)/\mathrm{RMSE}_b$, where the RMSE of a field is $\sqrt{\sum (\mathbf{u}-\mathbf{u}_t)^2 / \sum \mathbf{u}_t^2}$ over the domain at each advection step.

In [ ]:
fig = P.plot_rmse_time(cfg, rms, spans)
S.save_figure(fig, cfg, "fig2_rmse_time")
plt.show()

**Figure 2.** Velocity-field error over the assimilation window. Top strip: the correction windows of each method (overlapping sliding windows are drawn on separate rows). Middle: relative RMSE of the background (blue) and of each analysis. Bottom: RMSE reduction $100\,(\mathrm{RMSE}_b-\mathrm{RMSE}_a)/\mathrm{RMSE}_b$, labelled with its time mean. Vertical lines mark days.

In [ ]:
fig = P.plot_objective(cfg, obj)
S.save_figure(fig, cfg, "fig3_objective")
plt.show()

**Figure 3.** Decay of the objective function during the Adam iterations, one curve per window (later windows darker). The dot marks the final iteration of each window.

## 7 · Fields and trajectories in motion

Each movie shows three panels: background, analysis and truth. Every panel shows the observed (truth) tracks in ink with the panel's own trajectory overlaid. The strips below show the correction windows and the RMSE reduction. The movies play inline and are saved full-size to `output/notebooks/figures/<exp_name>/` (MP4 when ffmpeg is available, GIF otherwise).

The *analysis* fields are $\mathbf{u}_b + \delta\mathbf{u}$ with the **unfiltered** correction, as stored by `src/` and used for the RMSE. The analysis *trajectories* are advected in $\mathbf{u}_b + F(\delta\mathbf{u})$, the filtered correction that the cost function sees.

### 7a · Movie inputs

Fields at advection resolution for the sliding-window movie, and two extra trajectory sets: one advected in the background over the whole run, and one advected in the blended analysis. **Cost: moderate.**

The blended-analysis trajectory is an approximation. `simul_drifter` interpolates linearly between daily fields, so the blended field $\mathbf{u}_c(t)$ is sampled once per day. Because `src/` drops the final instant, the last day reuses the last available step.

In [ ]:
if "sliding" in cfg.methods:
    spd = cfg.steps_per_day
    with src_output(cfg, "movie_inputs"):
        Ub_s, Vb_s = (R.to_numpy(a).reshape(-1, *a.shape[2:]) for a in field_int_new(Ub, Vb, cfg.dt_advection))
        Ut_s, Vt_s = (R.to_numpy(a).reshape(-1, *a.shape[2:]) for a in field_int_new(Ut, Vt, cfg.dt_advection))
        Uc_s, Vc_s = R.load_fields(h5, "sliding", "Uc", "Vc")
        back_full = R.advect_all(cfg, Ub, Vb, lon, lat)
        Uc_d = torch.as_tensor(R.daily_from_steps(Uc_s, spd), device=device)
        Vc_d = torch.as_tensor(R.daily_from_steps(Vc_s, spd), device=device)
        anal_full = R.advect_all(cfg, Uc_d, Vc_d, lon, lat)
    print(f"{Uc_s.shape[0]} field snapshots, "
          f"{len(range(0, Uc_s.shape[0], cfg.frame_every))} movie frames")

### 7b · Constant correction per sub-window

One frame per sub-window. The fields are averaged over the sub-window, and the analysis adds that window's constant correction. Trajectory segments from earlier sub-windows stay visible, faded. **Cost: cheap to moderate.**

In [ ]:
if "standard" in cfg.methods:
    anim = M.movie_standard_windows(
        cfg, lon_np, lat_np, Ub_np, Vb_np, Ut_np, Vt_np,
        R.load_dU(h5, "standard", cfg), spans["standard"], obs,
        R.load_window_traj(h5, "standard", cfg, "back"),
        R.load_window_traj(h5, "standard", cfg, "last"),
        rms["standard"])
    path = M.save(anim, cfg, "movie_standard_windows", fps=1)
    if path:
        print("Saved", path.relative_to(cfg.root))
    display(M.show(anim))

**Movie 1.** Constant correction per sub-window. (a) Background, with drifters advected without correction. (b) Background plus the sub-window's correction, with drifters advected in the filtered correction. (c) Truth. Black lines are the observed tracks, and dots mark each drifter's position at the end of the sub-window. The window strip and the shaded band show which sub-window is displayed.

### 7c · Sliding window

One frame every `frame_every` advection steps. The fields are instantaneous. Tracks grow with time, and the faint grey line shows the path still to come. The window strip highlights the windows that contribute to the blended correction at the current time. **Cost: moderate** (rendering).

In [ ]:
if "sliding" in cfg.methods:
    anim = M.movie_sliding(
        cfg, lon_np, lat_np, Ub_s, Vb_s, Uc_s, Vc_s, Ut_s, Vt_s,
        spans["sliding"], obs, back_full, anal_full, rms["sliding"])
    path = M.save(anim, cfg, "movie_sliding")
    if path:
        print("Saved", path.relative_to(cfg.root))
    display(M.show(anim))

**Movie 2.** Sliding-window assimilation. (a) Background, with drifters advected in it over the whole run. (b) Blended analysis $\mathbf{u}_c(t)$, with drifters advected in its daily samples (approximation, see §7a). (c) Truth with the observed tracks. The cursor in the bottom strip follows the RMSE reduction at the current time.

## 8 · Finish

Closes the HDF5 file and lists what was written. Before committing the notebook, clear its outputs:

```bash
jupyter nbconvert --clear-output --inplace notebooks/assimilation.ipynb
```

In [ ]:
h5.close()
print("Results:", cfg.h5_path.relative_to(cfg.root))
if cfg.fig_dir.exists():
    for f in sorted(cfg.fig_dir.iterdir()):
        print("  ", f.relative_to(cfg.root))